# 고정 가중치 24상황 비교 — RL의 다음 기준 찾기

이 노트북은 **학습 날짜의 24개 상황 × 정책 5개**를 비교하고, 사전 규칙으로 선택한 고정 가중치를 기존 64조건에서 추가 비교합니다. 강화학습을 새로 실행하지 않습니다. 기존 결과는 수정하지 않습니다.

1. 런타임을 **CPU**로 선택하고 **런타임 → 모두 실행**을 누르세요.
2. Drive 접근을 허용하세요. 첫 상황의 5개 정책을 검사한 후 나머지를 자동 실행합니다.
3. 마지막에 결과가 나오면 저에게 “끝났어”라고 알려 주세요. 중간에 끊기면 다시 모두 실행하면 됩니다.

이번 실험은 약 **8.4작업시간**이 예상되지만 실제 속도에 따라 달라집니다. CPU 여러 개로 동시에 돌려도 작업시간은 합산합니다. 누적 96시간 한도에 가까우면 중단합니다. 모든 결과는 반복 사용된 날짜의 추가 개발 비교입니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE_ROOT / "FutureTraffic_FIXED24.zip"
SOURCE = DRIVE_ROOT / "explore_weight_v1"
PARENT = DRIVE_ROOT / "explore_weight_resume_v1"
NIGHT = DRIVE_ROOT / "explore_night_v1"
OUT = DRIVE_ROOT / "explore_fixed24_v1"
WORKSPACE = Path("/content/FutureTraffic_FIXED24")
WORK = Path("/content/fixed24_work")
for path in (ARCHIVE, SOURCE / "experiment.json", PARENT / "budget.json", NIGHT / "experiment.json"):
    assert path.is_file(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 결과 폴더:", OUT)

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib
import json
import stat
import sys
import types
import zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "WEIGHT_FIXED24_V1", "FIXED24 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델은 Drive 원본에서 읽어야 합니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("fixed24_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE}")

In [ ]:
import subprocess
import importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}
source_packages = json.loads((SOURCE / "experiment.json").read_text())["packages"]
assert all(source_packages.get(name) == version for name, version in EXPECTED.items()), "출발 실험의 패키지 버전이 이 묶음과 다릅니다."

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("출발 실험과 패키지 버전이 모두 같아 재설치를 건너뜁니다.")
else:
    print("출발 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 출발 실험과 다릅니다: {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
import subprocess
import signal
from IPython.display import display, Markdown
ENV = {**os.environ, "FOCUSED_RULESET": "pet1", "FOCUSED_NORTH_EXT": "240", "PYTHONUNBUFFERED": "1",
       "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
def run_live(args):
    process = subprocess.Popen([sys.executable, "-u", *args], cwd=WORKSPACE, env=ENV,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    try:
        for line in process.stdout:
            print(line.rstrip())
        code = process.wait()
    except KeyboardInterrupt:
        try:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait(timeout=20)
        except ProcessLookupError:
            pass
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
        raise
    if code != 0:
        raise RuntimeError(f"작업이 중단됐습니다 (종료 코드 {code}). 위의 원인을 확인해 주세요.")

run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_fixed24.py"])
print("실행 전 검사 통과")

In [ ]:
# 첫 5회도 본실험의 일부입니다. 완료된 작업은 다시 실행할 때 검증 후 재사용합니다.
try:
    run_live(["src/explore_fixed24.py", "--out", str(OUT), "--work", str(WORK),
              "--source", str(SOURCE), "--resume", str(PARENT), "--night", str(NIGHT), "--workers", "4"])
finally:
    summary = OUT / "summary.md"
    if summary.is_file():
        display(Markdown(summary.read_text()))
    print("결과 폴더:", OUT)